In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [27]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [28]:
#collecting data
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") #Sentinel-2 Surface Reflectance data

filtered_image = s2 \
    .filterBounds(region) \
    .sort('system:time_start')\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    #.filterDate('2022-01-01', '2022-12-31') \
    

print(f"Number of images found: {filtered_image.size().getInfo()}")

Number of images found: 327


In [29]:
#now making sure they have full coverage of the region
strict_collection = filtered_image.filter(ee.Filter.contains(
        leftField='.geo', #.geo refers to the geometry of the image
        rightValue=region #the region we defined earlier 
    )
)

print(f"Number of images with full coverage: {strict_collection.size().getInfo()}")

Number of images with full coverage: 325


In [30]:
min_timestamp = strict_collection.aggregate_min("system:time_start")
max_timestamp = strict_collection.aggregate_max("system:time_start")
print(f"Min Timestamp: {min_timestamp.getInfo()}")
print(f"Max Timestamp: {max_timestamp.getInfo()}")
print('First Date:', ee.Date(min_timestamp).format('YYYY-MM-dd').getInfo())
print('Last Date:', ee.Date(max_timestamp).format('YYYY-MM-dd').getInfo())

Min Timestamp: 1453092113893
Max Timestamp: 1766032934402
First Date: 2016-01-18
Last Date: 2025-12-18


In [31]:
# Visualize the collection
visualized_collection = strict_collection.map(lambda img: img.visualize(
    bands=['B4', 'B3', 'B2'],
    min=0,
    max=3000
))

In [32]:
import datetime

now = datetime.datetime.now()
time_stamp = now.strftime("%Y_%m_%d_%H_%M_%S")

file_name = '7 Trying to add dates to timelapse_' + time_stamp
print('Exporting file as:', file_name)

# Create the export task for video
task = ee.batch.Export.video.toDrive(
    collection=visualized_collection,
    folder='GEE_Exports',
    description=file_name,
    dimensions=720,    
    framesPerSecond=10, 
    region=region
)

task.start()
print(f"Export started with task ID: {task.id}")

Exporting file as: 7 Trying to add dates to timelapse_2026_01_07_05_23_48
Export started with task ID: QLDUTSWEKYIWXS5RZ3F2WSGX


In [33]:
# Wait for export to complete
from tqdm import tqdm
import time

try:
    with tqdm(desc="Exporting video", unit=" checks") as pbar:
        while task.active():
            pbar.update(1)
            time.sleep(30)
    print(f"Export complete and the video is saved in {task.status()['destination_uris'][0]}")
except KeyboardInterrupt:
    print('Export stopped')

Exporting video: 14 checks [07:14, 31.02s/ checks]


Export complete and the video is saved in https://drive.google.com/#folders/1PXNZcntSnrPKCShOF7h2WOVW7V9Qm6ex


In [34]:
#getting the list of dates from the collection
dates_list = strict_collection.aggregate_array('system:time_start').getInfo()

#convert timestamps to readable dates
from datetime import datetime
formatted_dates = [datetime.fromtimestamp(d/1000).strftime('%d-%m-%Y') for d in dates_list]

print(f"Total frames: {len(formatted_dates)}")
print(f"First date: {formatted_dates[0]}")
print(f"Last date: {formatted_dates[-1]}")

Total frames: 325
First date: 18-01-2016
Last date: 18-12-2025


In [35]:
#now downloading the video and keeping it in the same directory as this notebook

def add_dates_to_video(input_video_path, output_video_path, dates):
    import cv2
    
    cap = cv2.VideoCapture(input_video_path)
    
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"fps: {fps}, width: {width}, height: {height}, total_frames: {total_frames}")
    #just making sure the video properties are correct and match the number of dates
    print(f"Number of dates provided: {len(dates)}")
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    print(f"processing {total_frames} frames...")
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx < len(dates):
            date_text = dates[frame_idx]
        else:
            date_text = "Unknown"
        
        font = cv2.FONT_HERSHEY_SIMPLEX
        position = (30, 50)  # (x, y) from top-left
        font_scale = 1.5
        color = (255, 255, 255)  # White
        thickness = 3
        
        # add black outline for better visibility
        cv2.putText(frame, date_text, position, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(frame, date_text, position, font, font_scale, color, thickness)
        
        # nwo write the frame to output video
        out.write(frame)
        frame_idx += 1
        
        if frame_idx % 10 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames")
    
    cap.release()
    out.release()
    print(f"Video with dates saved to: {output_video_path}")

#testing
add_dates_to_video(file_name + '.mp4', file_name + '_with_dates.mp4', formatted_dates)

fps: 10, width: 720, height: 464, total_frames: 325
Number of dates provided: 325
processing 325 frames...
Processed 10/325 frames
Processed 20/325 frames
Processed 30/325 frames
Processed 40/325 frames
Processed 50/325 frames
Processed 60/325 frames
Processed 70/325 frames
Processed 80/325 frames
Processed 90/325 frames
Processed 100/325 frames
Processed 110/325 frames
Processed 120/325 frames
Processed 130/325 frames
Processed 140/325 frames
Processed 150/325 frames
Processed 160/325 frames
Processed 170/325 frames
Processed 180/325 frames
Processed 190/325 frames
Processed 200/325 frames
Processed 210/325 frames
Processed 220/325 frames
Processed 230/325 frames
Processed 240/325 frames
Processed 250/325 frames
Processed 260/325 frames
Processed 270/325 frames
Processed 280/325 frames
Processed 290/325 frames
Processed 300/325 frames
Processed 310/325 frames
Processed 320/325 frames
Video with dates saved to: 7 Trying to add dates to timelapse_2026_01_07_05_23_48_with_dates.mp4
